# View Evaluation Results

This notebook retrieves and visualizes evaluation run results from Agent Engine.

**Important**: Requires `pandas>=2.1.0` to avoid `TypeError: Cannot convert numpy.ndarray`
when constructing DataFrames from evaluation items containing numpy arrays.

In [ ]:
# Step 1: Upgrade pandas BEFORE importing vertexai
# This must run first and may require a runtime restart on Colab
%pip install --upgrade "pandas>=2.1.0" "numpy>=1.24.0" --quiet
%pip install --upgrade "google-cloud-aiplatform[evaluation]" "google-genai>=1.0.0" --quiet

In [ ]:
# Step 2: Configure project
PROJECT_ID = "wortz-project-352116"
LOCATION = "us-central1"

# Agent Engine IDs for each deployed agent
AGENT_ENGINE_IDS = {
    "grocery_assistant": "3727910666648944640",
    "mcp_analyst": "5787744546217525248",
    "simulator": "7053256041508634624",
    "a2a_agent": "2240491336593571840",
}

# Eval run IDs — populate these after running evaluations
# You can find eval run IDs in the Agent Engine console or from run_eval.py output
EVAL_RUNS = {
    "grocery_assistant": "7907494112918503424",
    # "mcp_analyst": "<eval_run_id>",
    # "simulator": "<eval_run_id>",
    # "a2a_agent": "<eval_run_id>",
}

In [ ]:
# Step 3: Authenticate (Colab only)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab")
except ImportError:
    print("Not on Colab — using Application Default Credentials")

In [ ]:
# Step 4: Initialize Vertex AI client
from vertexai import Client
from google.genai import types as genai_types

client = Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=genai_types.HttpOptions(api_version="v1beta1"),
)
print(f"Client initialized for {PROJECT_ID} in {LOCATION}")

In [ ]:
# Step 5: Retrieve and display evaluation results
for agent_name, eval_run_id in EVAL_RUNS.items():
    print(f"\n{'='*60}")
    print(f"Agent: {agent_name}")
    print(f"Eval Run: {eval_run_id}")
    print(f"{'='*60}")

    try:
        # Retrieve with evaluation items (requires pandas>=2.1.0)
        evaluation_run = client.evals.get_evaluation_run(
            name=eval_run_id,
            include_evaluation_items=True,
        )
        evaluation_run.show()
    except TypeError as e:
        if "numpy.ndarray" in str(e):
            print(f"\nDataFrame error (pandas version issue): {e}")
            print("Falling back to summary-only view...\n")

            # Fallback: retrieve without items
            evaluation_run = client.evals.get_evaluation_run(
                name=eval_run_id,
                include_evaluation_items=False,
            )
            print(f"Status: {evaluation_run.state}")
            print(f"Metrics: {evaluation_run.evaluation_run_results}")
        else:
            raise
    except Exception as e:
        print(f"Error retrieving eval run: {e}")